# 09 - Build Analytical Master Tables

This notebook builds the three stable analytical tables used by the exploratory analysis and reporting notebooks.

Final analytical outputs:

- `destinations_master.csv` supports Q1 on destination countries
- `entry_reasons_master.csv` supports Q2 on observable entry reasons
- `post_arrival_master.csv` supports Q3 on post-arrival trajectories

The goal is to create clean, reusable tables without combining indicators that measure different migration concepts.


## 1. Objective of the Notebook

The objective of this notebook is to transform standardized source-level datasets into final analytical master tables.

These master tables are designed to be:

- clean and reusable
- easy to inspect
- consistent in column naming
- suitable for analysis and visualization
- transparent about source and indicator differences

This notebook acts as the bridge between the standardization phase and the exploratory analysis phase.

## 2. Methodological Rules

The analytical tables keep different indicators separated.

This is a key methodological decision because the project uses multiple institutional data sources that do not measure the same phenomenon:

- UN DESA mainly measures migrant stock
- Eurostat first permits measure administrative entry permits
- Eurostat long-term residence measures durable residence status
- Eurostat citizenship acquisition measures a later administrative event
- OECD indicators provide complementary evidence with their own scope and definitions

For this reason, the project does not sum incompatible indicators together. Instead, each record keeps its `source`, `dataset`, `indicator`, `measure_type` and, when relevant, `sub_reason`.

This approach reduces the risk of misleading interpretation.

## 3. Import Libraries

This section imports the Python libraries required to build the analytical tables.

`pandas` is used to load, combine, filter and export structured datasets. `Path` is used to manage project paths in a cleaner and more reproducible way.

In [26]:
from pathlib import Path

import pandas as pd


## 4. Define Project Paths

This section defines the input and output folders used in the notebook.

Input files come from:

```text
data/processed/standardized/
```

Final analytical outputs are saved into:

```text
data/processed/analytical/
```

Using explicit paths helps make the notebook easier to understand and easier to run from the project repository.

In [27]:
PROJECT_ROOT = Path('..')

STANDARDIZED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'standardized'
ANALYTICAL_PATH = PROJECT_ROOT / 'data' / 'processed' / 'analytical'

ANALYTICAL_PATH.mkdir(parents=True, exist_ok=True)


## 5. Load Standardized Files

The datasets loaded in this section have already been cleaned and standardized in previous notebooks.

Each file is expected to contain harmonized columns such as:

- `source`
- `dataset`
- `origin_country`
- `destination_country`
- `year`
- `period`
- `indicator`
- `measure_type`
- `value`

The goal here is not to re-clean the raw data, but to assemble standardized files into analytical datasets.

In [28]:
undesa = pd.read_csv(STANDARDIZED_PATH / 'undesa_destinations_standardized.csv')
eurostat_first = pd.read_csv(STANDARDIZED_PATH / 'eurostat_first_permits_standardized.csv')
eurostat_long = pd.read_csv(STANDARDIZED_PATH / 'eurostat_long_term_residents_standardized.csv')
eurostat_status = pd.read_csv(STANDARDIZED_PATH / 'eurostat_status_changes_standardized.csv')
eurostat_citizenship = pd.read_csv(STANDARDIZED_PATH / 'eurostat_citizenship_acquisition_standardized.csv')
oecd = pd.read_csv(STANDARDIZED_PATH / 'oecd_migration_standardized.csv')


## 6. Helper Functions

This section defines small helper functions used to keep the analytical tables consistent.

The functions help:

- keep a clear column order
- avoid errors when optional columns are missing
- ensure that the `sub_reason` field exists when needed
- preserve traceability across different source datasets

These helper functions make the table-building process more robust and easier to maintain.

In [29]:
def keep_columns(df, columns):
    """Keep a clear column order while allowing optional columns to be absent."""
    return df[[col for col in columns if col in df.columns]].copy()


def ensure_sub_reason(df, default_value='not_applicable'):
    """Guarantee a stable Q2 schema for sources with or without sub-reasons."""
    df = df.copy()
    if 'sub_reason' not in df.columns:
        df['sub_reason'] = default_value
    else:
        missing = df['sub_reason'].isna() | (df['sub_reason'].astype('string').str.strip() == '')
        df.loc[missing, 'sub_reason'] = default_value
    return df


## 7. Build Q1 Analytical Table — Destination Countries

This section builds the table used to analyze the main destination countries of Cameroonian migrants.

The main source for this question is UN DESA, because it provides a global view of migrant stock by destination country.

OECD destination-related indicators are added only as complementary evidence. They remain clearly identified by their source and `measure_type`.

The output table supports the question:

**Q1: How did the main destination countries of Cameroonian migrants evolve across the 2015-2024 reference period?**

Important interpretation point: this table mainly captures migrant stock, not yearly migration flows.

In [30]:
oecd_destinations = oecd[oecd['analysis_question'].eq('Q1_destinations')].copy()

destinations_master = pd.concat([undesa, oecd_destinations], ignore_index=True)

destination_cols = [
    'source',
    'dataset',
    'origin_country',
    'destination_country',
    'year',
    'measure_type',
    'value',
    'total_migrants',
    'male_migrants',
    'female_migrants',
    'analysis_question',
]

destinations_master = keep_columns(destinations_master, destination_cols)
destinations_master


,source,dataset,origin_country,destination_country,year,measure_type,value,total_migrants,male_migrants,female_migrants,analysis_question
0,undesa,undesa_destinations,Cameroon,Zambia,2015,stock,214.0,214.0,114.0,100.0,Q1_destinations
1,undesa,undesa_destinations,Cameroon,Chad,2015,stock,30702.0,30702.0,13908.0,16794.0,Q1_destinations
2,undesa,undesa_destinations,Cameroon,Congo,2015,stock,11300.0,11300.0,6727.0,4573.0,Q1_destinations
3,undesa,undesa_destinations,Cameroon,Equatorial Guinea,2015,stock,798.0,798.0,535.0,263.0,Q1_destinations
4,undesa,undesa_destinations,Cameroon,Gabon,2015,stock,46269.0,46269.0,26905.0,19364.0,Q1_destinations
...,...,...,...,...,...,...,...,...,...,...,...
690,oecd,oecd_migration_database,Cameroon,Belgium,2018,stock_nationality,12828.0,NaN,NaN,NaN,Q1_destinations
691,oecd,oecd_migration_database,Cameroon,Belgium,2019,stock_nationality,13439.0,NaN,NaN,NaN,Q1_destinations
692,oecd,oecd_migration_database,Cameroon,Belgium,2020,stock_nationality,14229.0,NaN,NaN,NaN,Q1_destinations
693,oecd,oecd_migration_database,Cameroon,Belgium,2021,stock_nationality,14719.0,NaN,NaN,NaN,Q1_destinations


### 7.1 Quality Checks for the Destination Table

After building `destinations_master`, this section checks the structure of the output.

The checks focus on:

- table shape
- year coverage
- data sources included
- measure types available
- missing values in key fields

These checks confirm whether the table is ready for exploratory analysis and visualization.

In [31]:
print('Shape:', destinations_master.shape)
print('Years:', sorted(destinations_master['year'].dropna().unique()))
print('Sources:', destinations_master['source'].dropna().unique())
print('Measure types:', destinations_master['measure_type'].dropna().unique())


Shape: (695, 11)
Years: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2024)]
Sources: <StringArray>
['undesa', 'oecd']
Length: 2, dtype: str
Measure types: <StringArray>
['stock', 'inflow', 'stock_birth', 'stock_nationality']
Length: 4, dtype: str


## 8. Build Q2 Analytical Table — Observable Entry Reasons

This section builds the table used to analyze observable reasons for entry among Cameroonian migrants.

The main source is Eurostat first permits, because it provides administrative categories such as:

- family
- education
- work
- other reasons

OECD indicators related to worker inflows, seasonal workers or asylum-related inflows are included as complementary indicators when compatible with the analytical question.

The output table supports the question:

**Q2: How did the main observable entry reasons among Cameroonian migrants evolve across the available data period?**

Important interpretation point: first permits are administrative events. They should not be interpreted as total migrant stock.

In [32]:
eurostat_first = ensure_sub_reason(eurostat_first, default_value='not_applicable')
oecd_entry_reasons = ensure_sub_reason(
    oecd[oecd['analysis_question'].eq('Q2_entry_reasons')].copy(),
    default_value='not_applicable',
)

entry_reasons_master = pd.concat([eurostat_first, oecd_entry_reasons], ignore_index=True)

entry_cols = [
    'source',
    'dataset',
    'origin_country',
    'destination_country',
    'year',
    'reason',
    'sub_reason',
    'measure_type',
    'value',
    'analysis_question',
]

entry_reasons_master = keep_columns(entry_reasons_master, entry_cols)
entry_reasons_master.head()


,source,dataset,origin_country,destination_country,year,reason,sub_reason,measure_type,value,analysis_question
0,eurostat,eurostat_first_permits,Cameroon,Belgium,2015,family,not_applicable,permit,695.0,Q2_entry_reasons
1,eurostat,eurostat_first_permits,Cameroon,Bulgaria,2015,family,not_applicable,permit,0.0,Q2_entry_reasons
2,eurostat,eurostat_first_permits,Cameroon,Czechia,2015,family,not_applicable,permit,15.0,Q2_entry_reasons
3,eurostat,eurostat_first_permits,Cameroon,Denmark,2015,family,not_applicable,permit,55.0,Q2_entry_reasons
4,eurostat,eurostat_first_permits,Cameroon,Germany,2015,family,not_applicable,permit,748.0,Q2_entry_reasons


### 8.1 Quality Checks for the Entry Reasons Table

After building `entry_reasons_master`, this section verifies whether the output is analytically reliable.

The checks focus on:

- number of rows and columns
- year coverage
- source coverage
- measure types
- presence and consistency of `sub_reason`
- missing values in key columns

The `sub_reason` field is especially important because it preserves the reason category behind each entry-related indicator.

In [33]:
print('Shape:', entry_reasons_master.shape)
print('Years:', sorted(entry_reasons_master['year'].dropna().unique()))
print('Sources:', entry_reasons_master['source'].dropna().unique())
print('Measure types:', entry_reasons_master['measure_type'].dropna().unique())
print('Sub-reasons:', entry_reasons_master['sub_reason'].dropna().unique())


Shape: (1595, 10)
Years: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Sources: <StringArray>
['eurostat', 'oecd']
Length: 2, dtype: str
Measure types: <StringArray>
['permit', 'asylum_inflow', 'worker_inflow', 'seasonal_worker_inflow']
Length: 4, dtype: str
Sub-reasons: <StringArray>
['not_applicable', 'asylum', 'work', 'seasonal_work']
Length: 4, dtype: str


## 9. Build Q3 Analytical Table — Post-Arrival Trajectories

This section builds the table used to analyze post-arrival trajectories of Cameroonian migrants.

The table combines standardized indicators related to:

- long-term residence
- status changes
- citizenship acquisition
- selected OECD post-arrival indicators

These indicators are interpreted as administrative signals of post-arrival evolution, not as full personal migration journeys.

The output table supports the question:

**Q3: How did post-arrival trajectory indicators evolve across the available 2015-2024 data period?**

Important interpretation point: these indicators occur after arrival and should not be confused with first entry or migrant stock.

In [34]:
oecd_post_arrival = oecd[oecd['analysis_question'].eq('Q3_post_arrival_trajectories')].copy()

post_arrival_master = pd.concat(
    [eurostat_long, eurostat_status, eurostat_citizenship, oecd_post_arrival],
    ignore_index=True,
)

post_arrival_cols = [
    'source',
    'dataset',
    'origin_country',
    'destination_country',
    'year',
    'measure_type',
    'legal_framework',
    'value',
    'analysis_question',
]

post_arrival_master = keep_columns(post_arrival_master, post_arrival_cols)
post_arrival_master.head()


,source,dataset,origin_country,destination_country,year,measure_type,value,analysis_question
0,eurostat,eurostat_long_term_residents,Cameroon,Belgium,2015,long_term_resident,977.0,Q3_post_arrival_trajectories
1,eurostat,eurostat_long_term_residents,Cameroon,Bulgaria,2015,long_term_resident,6.0,Q3_post_arrival_trajectories
2,eurostat,eurostat_long_term_residents,Cameroon,Czechia,2015,long_term_resident,53.0,Q3_post_arrival_trajectories
3,eurostat,eurostat_long_term_residents,Cameroon,Denmark,2015,long_term_resident,121.0,Q3_post_arrival_trajectories
4,eurostat,eurostat_long_term_residents,Cameroon,Germany,2015,long_term_resident,80.0,Q3_post_arrival_trajectories


### 9.1 Quality Checks for the Post-Arrival Table

After building `post_arrival_master`, this section checks the reliability and structure of the output.

The checks focus on:

- table shape
- year coverage
- data sources
- measure types
- missing values
- consistency across post-arrival indicators

These checks help ensure that the table is ready for the exploratory analysis phase.

In [35]:
print('Shape:', post_arrival_master.shape)
print('Years:', sorted(post_arrival_master['year'].dropna().unique()))
print('Sources:', post_arrival_master['source'].dropna().unique())
print('Measure types:', post_arrival_master['measure_type'].dropna().unique())


Shape: (948, 8)
Years: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Sources: <StringArray>
['eurostat', 'oecd']
Length: 2, dtype: str
Measure types: <StringArray>
['long_term_resident', 'status_change', 'citizenship_acquisition']
Length: 3, dtype: str


## 10. Export Analytical Tables

This section exports the three final analytical master tables.

The exported files are:

```text
data/processed/analytical/destinations_master.csv
data/processed/analytical/entry_reasons_master.csv
data/processed/analytical/post_arrival_master.csv
```

These files become the stable inputs for the exploratory analysis and reporting notebooks.

In [36]:
destinations_master.to_csv(ANALYTICAL_PATH / 'destinations_master.csv', index=False)
entry_reasons_master.to_csv(ANALYTICAL_PATH / 'entry_reasons_master.csv', index=False)
post_arrival_master.to_csv(ANALYTICAL_PATH / 'post_arrival_master.csv', index=False)

print('Analytical tables exported successfully.')


Analytical tables exported successfully.


## 11. Methodological Limitations

The analytical tables should be interpreted with caution.

Main limitations:

- the sources do not always cover the same countries
- the available years differ depending on the source
- stock indicators and flow indicators are not directly comparable
- administrative indicators do not describe individual motivations
- post-arrival indicators do not reconstruct full migrant journeys

The strength of this notebook is that it keeps these differences visible instead of hiding them through inappropriate aggregation.

## 12. Summary

This notebook produces the three final analytical tables used in the project:

- `destinations_master.csv` supports Q1 on destination countries
- `entry_reasons_master.csv` supports Q2 on observable reasons for entry
- `post_arrival_master.csv` supports Q3 on post-arrival trajectories

These tables are now ready for the next phase of the project: exploratory analysis, interpretation and visual reporting.

By separating the tables by analytical question, the project remains methodologically clear and avoids mixing indicators that measure different migration concepts.